# 1D DSM in fully normalised vector spherical harmonics

Nobuaki Fuji 29/07/2026

Project the 3D radially transversely-isotropic elastic weak form onto one `(l,m)` multiplet before giving the radial equations to flexOPT.

In [1]:
# Locate flexOPT securely without relying on @__DIR__ (unreliable in IJulia).
import Pkg
function find_flexopt_root(start_dir=pwd())
    candidates = String[]
    haskey(ENV, "FLEXOPT_ROOT") && push!(candidates, abspath(expanduser(ENV["FLEXOPT_ROOT"])))
    directory = abspath(start_dir)
    while true
        push!(candidates, directory)
        parent = dirname(directory)
        parent == directory && break
        directory = parent
    end
    for candidate in unique(candidates)
        if isfile(joinpath(candidate, "Project.toml")) &&
           isfile(joinpath(candidate, "src", "commonBatchs.jl"))
            return candidate
        end
    end
    error("Cannot locate flexOPT. Start Jupyter inside the repository or set ENV[\"FLEXOPT_ROOT\"].")
end

flexopt_root = find_flexopt_root()
Pkg.activate(flexopt_root)
include(joinpath(flexopt_root, "src", "batchFiles", "batchSymbolics.jl"))
include(joinpath(flexopt_root, "src", "commonBatchs.jl"))
using .commonBatchs
include(joinpath(flexopt_root, "src", "planet1D.jl"))
planet1D.configure_input!()
using .planet1D
include(joinpath(flexopt_root, "src", "flexOPT.jl"))
using .flexOPT
using Symbolics, LinearAlgebra

  Activating project at `~/Documents/Github/flexOPT`


## Spherical surface projection rules

Let `L² = l(l+1)`, with `Y = Y_lm` fully normalised so that `∫Ω conj(Y)Y dΩ = 1`. Define the orthonormal vector harmonics

- `S¹ = Y e_r`,
- `S² = grad_Ω(Y)/L`,
- `T = -e_r × grad_Ω(Y)/L`.

The three harmonics are mutually orthogonal and have unit surface norm. The Hessian identities below are the only additional rules needed to integrate the strain energy. They make every projected coefficient independent of `m`.

In [2]:
# Symbolic record of the fully normalised surface integrals.
# H is the covariant Hessian on the unit sphere and g is its metric.
surface_integral_rules(L²) = Dict(
    :Y_Y                 => 1,
    :gradY_gradY         => L²,
    :S1_S1               => 1,
    :S2_S2               => 1,
    :T_T                 => 1,
    :cross_harmonics     => 0,
    :Y_laplacianY        => -L²,
    :H_H                 => L²*(L² - 1),
    :Y_gcolonH           => -L²,
    :symgradT_symgradT   => (L² - 2)/2,
)

surface_integral_rules (generic function with 1 method)

## Radial weak formulation

The TI symmetry axis is `e_r`. The five stored functions are `Crrrr`, `Crrtt = Crrθθ = Crrϕϕ`, `Ctttt = Cθθθθ = Cϕϕϕϕ`, `Cttpp = Cθθϕϕ`, and `Crtrt = Crθrθ = Crϕrϕ`. Tangential isotropy imposes `Cθϕθϕ = (Ctttt-Cttpp)/2`.

Write `u = U S¹ + V S² + W T` and use test amplitudes `(Ut,Vt,Wt)`. Introduce

- `D = U' + (2U-LV)/r` (dilatation),
- `A = V' + (LU-V)/r` (spheroidal radial shear),
- `B = W' - W/r` (toroidal radial shear).

After the sphere integral, the volume residual is `mass - stiffness`. The factor `r²` is retained because `dV = r² dr dΩ`. Natural boundary terms are `r² σ_rr Ut + r² σ_rS Vt + r² σ_rT Wt`.

In [7]:
@variables r ω　ℒ
@variables ρ(r) Crrrr(r) Crrtt(r) Ctttt(r) Cttpp(r) Crtrt(r) 
@variables Uₗₘ(r) Vₗₘ(r) Wₗₘ(r)
@variables Utest(r) Vtest(r) Wtest(r)
Dr = Differential(r)

dilatation(U, V, L) = Dr(U) + (2U - L*V)/r
radial_shear_S(U, V, L) = Dr(V) + (L*U - V)/r
radial_shear_T(W) = Dr(W) - W/r

function radial_weak_density(U, V, W, Ut, Vt, Wt, L)
    L² = L^2
    D, Dt = dilatation(U,V,L), dilatation(Ut,Vt,L)
    A, At = radial_shear_S(U,V,L), radial_shear_S(Ut,Vt,L)
    B, Bt = radial_shear_T(W), radial_shear_T(Wt)

    τ, τt = (2U-L*V)/r, (2Ut-L*Vt)/r
    Kt = (Ctttt + Cttpp)/2
    H = (Ctttt - Cttpp)*(L²-2)/2
    stiffness = r^2*(Crrrr*Dr(U)*Dr(Ut) + Crrtt*(Dr(U)*τt + τ*Dr(Ut)) +
        Kt*τ*τt + Crtrt*(A*At + B*Bt)) + H*(V*Vt + W*Wt)
    mass = r^2*ρ*ω^2*(U*Ut + V*Vt + W*Wt)
    return mass - stiffness
end

weak_density = radial_weak_density(Uₗₘ,Vₗₘ,Wₗₘ,Utest,Vtest,Wtest,√6) # l=2 example

-1.9999999999999996(-Cttpp(r) + Ctttt(r))*(Vtest(r)*Vₗₘ(r) + Wtest(r)*Wₗₘ(r)) - (r^2)*(((1//2)*(Cttpp(r) + Ctttt(r))*(-2.449489742783178Vtest(r) + 2Utest(r))*(-2.449489742783178Vₗₘ(r) + 2Uₗₘ(r))) / (r^2) + Crrtt(r)*(((-2.449489742783178Vₗₘ(r) + 2Uₗₘ(r))*Differential(r)(Utest(r))) / r + ((-2.449489742783178Vtest(r) + 2Utest(r))*Differential(r)(Uₗₘ(r))) / r) + ((Differential(r)(Vₗₘ(r)) + (-Vₗₘ(r) + 2.449489742783178Uₗₘ(r)) / r)*(Differential(r)(Vtest(r)) + (-Vtest(r) + 2.449489742783178Utest(r)) / r) + ((-Wₗₘ(r)) / r + Differential(r)(Wₗₘ(r)))*(Differential(r)(Wtest(r)) + (-Wtest(r)) / r))*Crtrt(r) + Differential(r)(Uₗₘ(r))*Crrrr(r)*Differential(r)(Utest(r))) + (r^2)*(Vtest(r)*Vₗₘ(r) + Wtest(r)*Wₗₘ(r) + Uₗₘ(r)*Utest(r))*ρ(r)*(ω^2)

## Radial equations supplied to flexOPT

`radial_exprs(l)` returns the projected strong residual but deliberately preserves each radial flux as an explicit outer `Dr(flux)`. In `variationalForm=:natural_weak`, flexOPT moves exactly those derivatives onto its test function. No angular symbol occurs in these expressions. All `m` for a fixed `l` share the same operator.

In [8]:
function radial_exprs(L)
    L² = L^2
    U, V, W = Uₗₘ, Vₗₘ, Wₗₘ
    τ = (2U-L*V)/r
    A = radial_shear_S(U,V,L)
    B = radial_shear_T(W)
    Kt = (Ctttt + Cttpp)/2
    H = (Ctttt - Cttpp)*(L²-2)/2

    # Radial traction amplitudes; these become the natural boundary fluxes.
    P_U = r^2*(Crrrr*Dr(U) + Crrtt*τ)
    P_V = r^2*Crtrt*A
    P_W = r^2*Crtrt*B

    R_U = Dr(P_U) - 2r*Crrtt*Dr(U) - 2r*Kt*τ - r*L*Crtrt*A + r^2*ρ*ω^2*U
    R_V = Dr(P_V) + r*L*Crrtt*Dr(U) + r*L*Kt*τ + r*Crtrt*A - H*V + r^2*ρ*ω^2*V
    R_W = Dr(P_W) + r*Crtrt*B - H*W + r^2*ρ*ω^2*W
    return (R_U, R_V, R_W)
end

exprs = radial_exprs(ℒ)       # fill ℒ=sqrt(l(l+1)) for each solve
fields = (Uₗₘ, Vₗₘ, Wₗₘ)
vars = (ρ, Crrrr, Crrtt, Ctttt, Cttpp, Crtrt, ℒ, ω)
coordinates = (r,)
∂, ∂² = usefulPartials(coordinates)
equationCharacteristics = (; exprs, fields, vars, coordinates, ∂, ∂²)

(exprs = (Differential(r)((r^2)*((Crrtt(r)*(2Uₗₘ(r) - Vₗₘ(r)*ℒ)) / r + Differential(r)(Uₗₘ(r))*Crrrr(r))) - 2r*Crrtt(r)*Differential(r)(Uₗₘ(r)) - (Cttpp(r) + Ctttt(r))*(2Uₗₘ(r) - Vₗₘ(r)*ℒ) - r*(Differential(r)(Vₗₘ(r)) + (-Vₗₘ(r) + Uₗₘ(r)*ℒ) / r)*Crtrt(r)*ℒ + (r^2)*ρ(r)*Uₗₘ(r)*(ω^2), Differential(r)((r^2)*(Differential(r)(Vₗₘ(r)) + (-Vₗₘ(r) + Uₗₘ(r)*ℒ) / r)*Crtrt(r)) + r*(Differential(r)(Vₗₘ(r)) + (-Vₗₘ(r) + Uₗₘ(r)*ℒ) / r)*Crtrt(r) + r*Crrtt(r)*Differential(r)(Uₗₘ(r))*ℒ - (1//2)*(-Cttpp(r) + Ctttt(r))*Vₗₘ(r)*(-2 + ℒ^2) + (1//2)*(Cttpp(r) + Ctttt(r))*(2Uₗₘ(r) - Vₗₘ(r)*ℒ)*ℒ + (r^2)*ρ(r)*Vₗₘ(r)*(ω^2), Differential(r)((r^2)*((-Wₗₘ(r)) / r + Differential(r)(Wₗₘ(r)))*Crtrt(r)) + r*((-Wₗₘ(r)) / r + Differential(r)(Wₗₘ(r)))*Crtrt(r) - (1//2)*(-Cttpp(r) + Ctttt(r))*Wₗₘ(r)*(-2 + ℒ^2) + (r^2)*ρ(r)*Wₗₘ(r)*(ω^2)), fields = (Uₗₘ(r), Vₗₘ(r), Wₗₘ(r)), vars = (ρ(r), Crrrr(r), Crrtt(r), Ctttt(r), Cttpp(r), Crtrt(r), ℒ, ω), coordinates = (r,), ∂ = Any[Differential(r)], ∂² = Any[Differential(r) ∘ Different

In [5]:
# Confirm that flexOPT sees one undifferentiated and one radial-test-derivative group.
weak_form = naturalWeakForm(equationCharacteristics)
weak_groups = weakTermGroups(weak_form)
@assert sort(collect(keys(weak_groups))) == [(0,), (1,)]
@assert all(!occursin(s, string(exprs)) for s in ("θ", "ϕ", "Yₗₘ"))
(; weak_groups, boundary_fluxes=weak_form.boundary_fluxes)

(weak_groups = Dict{Any, Vector{Any}}((0,) => [-2r*Crrtt(r)*Differential(r)(Uₗₘ(r)) - (Cttpp(r) + Ctttt(r))*(2Uₗₘ(r) - ℒ(r)*Vₗₘ(r)) - r*((-Vₗₘ(r) + ℒ(r)*Uₗₘ(r)) / r + Differential(r)(Vₗₘ(r)))*ℒ(r)*Crtrt(r) + (r^2)*ρ(r)*Uₗₘ(r)*(ω^2), r*((-Vₗₘ(r) + ℒ(r)*Uₗₘ(r)) / r + Differential(r)(Vₗₘ(r)))*Crtrt(r) + r*ℒ(r)*Crrtt(r)*Differential(r)(Uₗₘ(r)) - (1//2)*(-Cttpp(r) + Ctttt(r))*(-2 + ℒ(r)^2)*Vₗₘ(r) + (1//2)*(Cttpp(r) + Ctttt(r))*ℒ(r)*(2Uₗₘ(r) - ℒ(r)*Vₗₘ(r)) + (r^2)*ρ(r)*Vₗₘ(r)*(ω^2), r*((-Wₗₘ(r)) / r + Differential(r)(Wₗₘ(r)))*Crtrt(r) - (1//2)*(-Cttpp(r) + Ctttt(r))*(-2 + ℒ(r)^2)*Wₗₘ(r) + (r^2)*ρ(r)*Wₗₘ(r)*(ω^2)], (1,) => [-(r^2)*(((2Uₗₘ(r) - ℒ(r)*Vₗₘ(r))*Crrtt(r)) / r + Differential(r)(Uₗₘ(r))*Crrrr(r)), -(r^2)*((-Vₗₘ(r) + ℒ(r)*Uₗₘ(r)) / r + Differential(r)(Vₗₘ(r)))*Crtrt(r), -(r^2)*((-Wₗₘ(r)) / r + Differential(r)(Wₗₘ(r)))*Crtrt(r)]), boundary_fluxes = (BoundaryFlux[BoundaryFlux{SymbolicUtils.BasicSymbolic{Real}, 1}(-(r^2)*(((2Uₗₘ(r) - ℒ(r)*Vₗₘ(r))*Crrtt(r)) / r + Differential(r)(Uₗₘ(r))*C

In [6]:
# l=0 has only U00; its radial flux is still visible to :natural_weak.
function radial_expr_l0()
    U = Uₗₘ
    τ = 2U/r
    Kt = (Ctttt + Cttpp)/2
    P_U = r^2*(Crrrr*Dr(U) + Crrtt*τ)
    return Dr(P_U) - 2r*Crrtt*Dr(U) - 2r*Kt*τ + r^2*ρ*ω^2*U
end

equationCharacteristics_l0 = (;
    exprs=(radial_expr_l0(),), fields=(Uₗₘ,),
    vars=(ρ,Crrrr,Crrtt,Ctttt,Cttpp,ω), coordinates, ∂, ∂²)

# Y_lm and its tangent-plane gradient/Hessian at the north pole.
# Complex, fully normalised, Condon-Shortley convention.
function pole_harmonic_jet(l::Integer, m::Integer)
    q = l*(l+1)
    a = sqrt((2l+1)/(4π))
    z2 = zeros(ComplexF64,2); Z2 = zeros(ComplexF64,2,2)
    abs(m) > min(l,2) && return (Y=0.0+0im, grad=z2, Hess=Z2)
    if m == 0
        return (Y=complex(a), grad=z2, Hess=ComplexF64[-q*a/2 0; 0 -q*a/2])
    elseif m == 1
        c = -a*sqrt(q)/2
        return (Y=0.0+0im, grad=ComplexF64[c,im*c], Hess=Z2)
    elseif m == -1
        c = a*sqrt(q)/2
        return (Y=0.0+0im, grad=ComplexF64[c,-im*c], Hess=Z2)
    else
        c = a*sqrt(q*(q-2))/8
        s = m == 2 ? 1 : -1
        return (Y=0.0+0im, grad=z2, Hess=ComplexF64[2c s*2im*c; s*2im*c -2c])
    end
end

# Coefficients multiplying (test, Dr(test)) in M:ε(test*) at source radius rs.
function north_pole_double_couple(l::Integer, m::Integer, M::AbstractMatrix, rs::Real)
    size(M) == (3,3) || throw(DimensionMismatch("M uses (er,ex,ey) components"))
    isapprox(M,transpose(M)) || throw(ArgumentError("moment tensor must be symmetric"))
    isapprox(tr(M),zero(eltype(M)); atol=100eps(Float64)*max(1,norm(M))) ||
        throw(ArgumentError("a double couple must be trace-free"))
    j = pole_harmonic_jet(l,m); q=l*(l+1); L=sqrt(q)
    contract(E) = sum(M .* conj.(E))
    function pair(E0,E1); (value=contract(E0), derivative=contract(E1)); end

    EU0=zeros(ComplexF64,3,3); EU1=zeros(ComplexF64,3,3)
    EU1[1,1]=j.Y; EU0[2,2]=EU0[3,3]=j.Y/rs
    EU0[1,2:3]=j.grad/(2rs); EU0[2:3,1]=j.grad/(2rs)
    if l == 0
        return (U=pair(EU0,EU1), V=(value=0im,derivative=0im), W=(value=0im,derivative=0im))
    end

    EV0=zeros(ComplexF64,3,3); EV1=zeros(ComplexF64,3,3)
    EV1[1,2:3]=j.grad/(2L); EV1[2:3,1]=j.grad/(2L)
    EV0[1,2:3]=-j.grad/(2L*rs); EV0[2:3,1]=-j.grad/(2L*rs)
    EV0[2:3,2:3]=j.Hess/(L*rs)

    J=ComplexF64[0 1;-1 0]; t=J*j.grad/L; dt=J*j.Hess/L
    EW0=zeros(ComplexF64,3,3); EW1=zeros(ComplexF64,3,3)
    EW1[1,2:3]=t/2; EW1[2:3,1]=t/2
    EW0[1,2:3]=-t/(2rs); EW0[2:3,1]=-t/(2rs)
    EW0[2:3,2:3]=(dt+transpose(dt))/(2rs)
    return (U=pair(EU0,EU1), V=pair(EV0,EV1), W=pair(EW0,EW1))
end

# PREM polynomial evaluation and conversion to the five complex radial-TI moduli.
poly4(c,x) = c[1]+c[2]*x+c[3]*x^2+c[4]*x^3
constant_Q_factor(Q; sign=-1) = Q > 0 ? 1 + sign*im/Q : 1 + 0im
function prem_ti_at(model, x; attenuation_sign=-1)
    i = findfirst(k -> model.normalisedBottomRadius[k] <= x <= model.normalisedTopRadius[k], 1:model.nzone)
    isnothing(i) && throw(BoundsError(model.normalisedBottomRadius,x))
    val(C)=poly4(view(C,i,:),x)
    density=val(model.C_ρ); vpv=val(model.C_Vpv); vph=val(model.C_Vph)
    vsv=val(model.C_Vsv); vsh=val(model.C_Vsh); eta=val(model.C_η)
    fκ=constant_Q_factor(val(model.C_Qκ); sign=attenuation_sign)
    fμ=constant_Q_factor(val(model.C_Qμ); sign=attenuation_sign)
    A0=density*vph^2; C0=density*vpv^2; L0=density*vsv^2; N0=density*vsh^2
    N=N0*fμ; L=L0*fμ
    A=(A0-4N0/3)*fκ + 4N0*fμ/3
    C=(C0-4L0/3)*fκ + 4L0*fμ/3
    F=eta*(A-2L)
    return (ρ=density,Crrrr=C,Crrtt=F,Ctttt=A,Cttpp=A-2N,Crtrt=L)
end

# Selection rule audit: a point moment uses at most second angular derivatives.
Mdc = ComplexF64[1 0 0; 0 -1 0; 0 0 0] # example DC in the pole frame
@assert all(north_pole_double_couple(4,m,Mdc,0.9) ==
            (U=(value=0im,derivative=0im),V=(value=0im,derivative=0im),W=(value=0im,derivative=0im))
            for m in (-4,-3,3,4))